# NLP Lab 3: Regex, Token Attributes, Tokenizers and Sentence Segmentation

| Activity | Topic |
|---|---|
| 1 | Extract emails, order numbers and phone numbers with **regex** |
| 2 | Extract currency amounts and numeric quantities with **spaCy token attributes** (no regex) and compare with a regex approach |
| 3 | Tokenize a document with **NLTK** and **spaCy** and note the differences (punctuation, contractions, ...) |
| 4 | Segment a 6-sentence paragraph with the **NLTK** and **spaCy** sentence splitters and flag mismatches |

> The sample text block and document below are written for this notebook. To use your own data, just replace the `RAW_TEXT`, `DOCUMENT` and `PARAGRAPH` variables; every cell recomputes its results.

In [1]:
%pip install -q nltk spacy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import difflib

import nltk
import spacy
from nltk.tokenize import word_tokenize, sent_tokenize

# NLTK tokenizer data (downloaded only if missing)
for res, path in [("punkt", "tokenizers/punkt"), ("punkt_tab", "tokenizers/punkt_tab")]:
    try:
        nltk.data.find(path)
    except LookupError:
        nltk.download(res, quiet=True)

# spaCy: a blank English pipeline already contains the rule-based tokenizer and the
# lexical token attributes used in this lab (like_num, is_currency, like_email, ...).
nlp = spacy.blank("en")

print("NLTK  version:", nltk.__version__)
print("spaCy version:", spacy.__version__)


def print_table(rows, headers):
    # tiny helper: print a left-aligned text table
    rows = [[str(c) for c in r] for r in rows]
    widths = [max(len(h), *(len(r[i]) for r in rows)) if rows else len(h) for i, h in enumerate(headers)]
    line = "  ".join(h.ljust(w) for h, w in zip(headers, widths))
    print(line)
    print("  ".join("-" * w for w in widths))
    for r in rows:
        print("  ".join(c.ljust(w) for c, w in zip(r, widths)))


def norm(s):
    # normalise whitespace/case so strings from different methods can be compared
    return re.sub(r"\s+", " ", s).strip().lower()

NLTK  version: 3.10.4
spaCy version: 3.8.16


## The raw text block (used in Activities 1 and 2)

In [3]:
RAW_TEXT = '''
Subject: Order confirmation and delivery update

Hi team,

Customer Ayesha Khan (ayesha.khan@example.com) placed order #ORD-20481 on 12 March. She ordered 3 units of the Pro Keyboard at $45.50 each and 2 pcs of USB-C cables at Rs. 1,250 each, for a total of $136.00 (approx. Rs. 38,000). Please call her on +92 300 1234567 or (555) 123-4567 if the courier misses delivery.

A second customer, Bilal Ahmed, wrote from bilal_ahmed99@mail.example.org about Order No. 77123 and Order ID: A-55902. He asked for 12 items, 2.5 kg of packing material, and a refund of €30 plus £9.99 for shipping. His numbers are 0300-5551234 and +1-415-555-0199. Please also cc support@shop.co.uk and sales.team@shop.co.uk.

Shipment of five boxes is due in 10 days. Pricing: USD 1,200 for bulk orders over 1,000 units.
'''.strip()

print(RAW_TEXT)

Subject: Order confirmation and delivery update

Hi team,

Customer Ayesha Khan (ayesha.khan@example.com) placed order #ORD-20481 on 12 March. She ordered 3 units of the Pro Keyboard at $45.50 each and 2 pcs of USB-C cables at Rs. 1,250 each, for a total of $136.00 (approx. Rs. 38,000). Please call her on +92 300 1234567 or (555) 123-4567 if the courier misses delivery.

A second customer, Bilal Ahmed, wrote from bilal_ahmed99@mail.example.org about Order No. 77123 and Order ID: A-55902. He asked for 12 items, 2.5 kg of packing material, and a refund of €30 plus £9.99 for shipping. His numbers are 0300-5551234 and +1-415-555-0199. Please also cc support@shop.co.uk and sales.team@shop.co.uk.

Shipment of five boxes is due in 10 days. Pricing: USD 1,200 for bulk orders over 1,000 units.


## Activity 1: Regex extraction of emails, order numbers and phone numbers

Patterns used:

| Entity | Idea |
|---|---|
| Email | `local-part @ domain . tld`; the local part may contain letters, digits and `. _ % + -` |
| Order number | the id that follows `#`, `Order No.`, `Order ID:` (an optional letter prefix such as `ORD-` or `A-`, then 4+ digits) |
| Phone number | three formats: international (`+92 300 1234567`, `+1-415-555-0199`), US style (`(555) 123-4567`) and local (`0300-5551234`) |

In [4]:
EMAIL_RE = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")

ORDER_RE = re.compile(r"(?:#|\bOrder\s+(?:No\.?|ID:?)\s*)((?:[A-Z]{1,4}-)?\d{4,})")

PHONE_RE = re.compile(
    r"(?<![\w+])"
    r"(?:"
    r"\+\d{1,3}[\s-]?\d{2,4}[\s-]?\d{3,4}[\s-]?\d{3,4}"   # international
    r"|\(\d{3}\)\s?\d{3}-\d{4}"                            # (555) 123-4567
    r"|0\d{3}-\d{7}"                                       # 0300-5551234
    r")"
    r"(?!\d)"
)

emails = EMAIL_RE.findall(RAW_TEXT)
orders = [m.group(1) for m in ORDER_RE.finditer(RAW_TEXT)]
phones = PHONE_RE.findall(RAW_TEXT)

print("EMAILS")
for e in emails:
    print("  ", e)
print("\nORDER NUMBERS")
for o in orders:
    print("  ", o)
print("\nPHONE NUMBERS")
for p in phones:
    print("  ", p)

EMAILS
   ayesha.khan@example.com
   bilal_ahmed99@mail.example.org
   support@shop.co.uk
   sales.team@shop.co.uk

ORDER NUMBERS
   ORD-20481
   77123
   A-55902

PHONE NUMBERS
   +92 300 1234567
   (555) 123-4567
   0300-5551234
   +1-415-555-0199


In [5]:
# where in the text was each entity found? (character offsets via finditer)
rows = []
for label, rx, grp in [("email", EMAIL_RE, 0), ("order", ORDER_RE, 1), ("phone", PHONE_RE, 0)]:
    for m in rx.finditer(RAW_TEXT):
        rows.append((label, m.group(grp), m.start(grp), m.end(grp)))
rows.sort(key=lambda r: r[2])
print_table(rows, ["type", "match", "start", "end"])

print("\nCounts:", {"emails": len(emails), "orders": len(orders), "phones": len(phones)})

type   match                           start  end
-----  ------------------------------  -----  ---
email  ayesha.khan@example.com         81     104
order  ORD-20481                       120    129
phone  +92 300 1234567                 307    322
phone  (555) 123-4567                  326    340
email  bilal_ahmed99@mail.example.org  417    447
order  77123                           464    469
order  A-55902                         484    491
phone  0300-5551234                    605    617
phone  +1-415-555-0199                 622    637
email  support@shop.co.uk              654    672
email  sales.team@shop.co.uk           677    698

Counts: {'emails': 4, 'orders': 3, 'phones': 4}


In [6]:
# sanity check against the answers we expect for this text block
expected = {
    "emails": ["ayesha.khan@example.com", "bilal_ahmed99@mail.example.org",
               "support@shop.co.uk", "sales.team@shop.co.uk"],
    "orders": ["ORD-20481", "77123", "A-55902"],
    "phones": ["+92 300 1234567", "(555) 123-4567", "0300-5551234", "+1-415-555-0199"],
}
found = {"emails": emails, "orders": orders, "phones": phones}
for key in expected:
    print(f"{key:<7} all found, nothing extra: {sorted(found[key]) == sorted(expected[key])}")

emails  all found, nothing extra: True
orders  all found, nothing extra: True
phones  all found, nothing extra: True


## Activity 2: Currency amounts and numeric quantities with spaCy token attributes (no regex)

spaCy's tokenizer attaches boolean attributes to every token. The ones used here:

- `token.is_currency`: is it a currency symbol (`$`, `€`, `£`, ...)?
- `token.like_num`: does it look like a number (`45.50`, `1,250`, `five`)?
- `token.lower_`: lower-cased text, used to look up currency codes (`usd`, `rs.`) and unit words (`kg`, `units`)

**Rules (attributes only):**
1. *Currency amount*: a token that `is_currency` (or is a known currency code) immediately followed by a `like_num` token.
2. *Quantity*: a `like_num` token (not part of a currency amount) followed by a unit/noun token such as `units`, `pcs`, `kg`, `items`, `boxes`, `days`.

In [7]:
doc = nlp(RAW_TEXT)

# Look at the attributes of a few interesting tokens
probe = [t for t in doc if t.is_currency or t.like_num][:18]
print_table([(t.text, t.is_currency, t.like_num, t.is_digit, t.is_alpha) for t in probe],
            ["token", "is_currency", "like_num", "is_digit", "is_alpha"])

token    is_currency  like_num  is_digit  is_alpha
-------  -----------  --------  --------  --------
12       False        True      True      False   
3        False        True      True      False   
$        True         False     False     False   
45.50    False        True      False     False   
2        False        True      True      False   
1,250    False        True      False     False   
$        True         False     False     False   
136.00   False        True      False     False   
38,000   False        True      False     False   
+92      False        True      False     False   
300      False        True      True      False   
1234567  False        True      True      False   
555      False        True      True      False   
123      False        True      True      False   
4567     False        True      True      False   
second   False        True      False     True    
77123    False        True      True      False   
12       False        True     

In [8]:
CURRENCY_CODES = {"usd", "pkr", "rs", "eur", "gbp"}
UNITS = {"unit", "units", "pc", "pcs", "kg", "item", "items", "box", "boxes", "day", "days"}


def spacy_currency(doc):
    # currency symbol/code token, optionally followed by a '.' token (spaCy splits "Rs." into "Rs" + "."),
    # followed by a number-like token
    amounts, used = [], set()
    for i, tok in enumerate(doc):
        if tok.is_currency or tok.lower_ in CURRENCY_CODES:
            j = i + 1
            if j < len(doc) and doc[j].text == ".":
                j += 1
            if j < len(doc) and doc[j].like_num:
                amounts.append(doc[i:j + 1].text)      # Span.text keeps the original spacing
                used.update(range(i, j + 1))
    return amounts, used


def spacy_quantities(doc, used):
    # number-like token (not already used by a currency amount) followed by a unit word
    quantities, idx = [], set()
    for i in range(len(doc) - 1):
        if i in used:
            continue
        if doc[i].like_num and doc[i + 1].lower_ in UNITS:
            quantities.append(doc[i:i + 2].text)
            idx.update({i, i + 1})
    return quantities, idx


sp_currency, used_idx = spacy_currency(doc)
sp_quantity, quantity_idx = spacy_quantities(doc, used_idx)

print("CURRENCY (spaCy attributes):", sp_currency)
print("QUANTITIES (spaCy attributes):", sp_quantity)

# like_num tokens that are neither currency nor quantity: what a naive "like_num only" rule would also flag
leftover = [t.text for i, t in enumerate(doc) if t.like_num and i not in used_idx and i not in quantity_idx]
print("\nOther like_num tokens (dates, order ids, phone fragments, ordinals):", leftover)

CURRENCY (spaCy attributes): ['$45.50', 'Rs. 1,250', '$136.00', 'Rs. 38,000', '€30', '£9.99', 'USD 1,200']
QUANTITIES (spaCy attributes): ['3 units', '2 pcs', '12 items', '2.5 kg', 'five boxes', '10 days', '1,000 units']

Other like_num tokens (dates, order ids, phone fragments, ordinals): ['12', '+92', '300', '1234567', '555', '123', '4567', 'second', '77123', '0300', '5551234', '+1', '415', '555', '0199']


In [9]:
# The same two tasks with regex, for comparison
CURRENCY_RE = re.compile(
    r"(?:[$€£]\s?\d[\d,]*(?:\.\d+)?"
    r"|(?:USD|PKR|EUR|GBP|Rs\.?)\s?\d[\d,]*(?:\.\d+)?)"
)
NUMWORDS = r"one|two|three|four|five|six|seven|eight|nine|ten"
QUANTITY_RE = re.compile(
    r"\b(?:\d[\d,]*(?:\.\d+)?|" + NUMWORDS + r")\s*(?:units?|pcs?|kg|items?|boxes|days?)\b",
    re.IGNORECASE,
)

re_currency = CURRENCY_RE.findall(RAW_TEXT)
re_quantity = QUANTITY_RE.findall(RAW_TEXT)

print("CURRENCY (regex):  ", re_currency)
print("QUANTITIES (regex):", re_quantity)

CURRENCY (regex):   ['$45.50', 'Rs. 1,250', '$136.00', 'Rs. 38,000', '€30', '£9.99', 'USD 1,200']
QUANTITIES (regex): ['3 units', '2 pcs', '12 items', '2.5 kg', 'five boxes', '10 days', '1,000 units']


In [10]:
gold_currency = ["$45.50", "Rs. 1,250", "$136.00", "Rs. 38,000", "€30", "£9.99", "USD 1,200"]
gold_quantity = ["3 units", "2 pcs", "12 items", "2.5 kg", "five boxes", "10 days", "1,000 units"]


def compare(name, spacy_out, regex_out, gold):
    s, r, g = {norm(x) for x in spacy_out}, {norm(x) for x in regex_out}, {norm(x) for x in gold}

    def prf(found):
        tp = len(found & g)
        prec = tp / len(found) if found else 0.0
        rec = tp / len(g) if g else 0.0
        return round(prec, 4), round(rec, 4)

    print(f"=== {name} ===")
    print("found by both         :", sorted(s & r))
    print("only spaCy attributes :", sorted(s - r))
    print("only regex            :", sorted(r - s))
    print("missed by spaCy       :", sorted(g - s))
    print("missed by regex       :", sorted(g - r))
    print_table([("spaCy attributes", *prf(s)), ("regex", *prf(r))], ["method", "precision", "recall"])
    print()


compare("Currency amounts", sp_currency, re_currency, gold_currency)
compare("Numeric quantities", sp_quantity, re_quantity, gold_quantity)

=== Currency amounts ===
found by both         : ['$136.00', '$45.50', 'rs. 1,250', 'rs. 38,000', 'usd 1,200', '£9.99', '€30']
only spaCy attributes : []
only regex            : []
missed by spaCy       : []
missed by regex       : []
method            precision  recall
----------------  ---------  ------
spaCy attributes  1.0        1.0   
regex             1.0        1.0   

=== Numeric quantities ===
found by both         : ['1,000 units', '10 days', '12 items', '2 pcs', '2.5 kg', '3 units', 'five boxes']
only spaCy attributes : []
only regex            : []
missed by spaCy       : []
missed by regex       : []
method            precision  recall
----------------  ---------  ------
spaCy attributes  1.0        1.0   
regex             1.0        1.0   



### Comparison: attributes vs regex

Both methods find all currency amounts and all quantities in this text block. One detail is worth noting: spaCy's tokenizer splits `Rs.` into two tokens (`Rs`, `.`), so the attribute rule has to allow an optional `.` token between the currency code and the number. Without it, `Rs. 1,250` and `Rs. 38,000` are missed. The check that follows `like_num` with a unit word is what keeps dates (`12 March`), order ids, phone-number fragments and even the ordinal `second` (which also has `like_num = True`) out of the quantity list.

| | spaCy token attributes | Regex |
|---|---|---|
| Number formats | `like_num` already understands `1,250`, `45.50` and even words such as `five` | every format (decimals, thousands separators, number words) must be written into the pattern |
| Currency | `is_currency` knows many symbols; currency *codes* (`USD`, `Rs.`) need a small lookup set | symbols and codes are listed explicitly in the pattern |
| Context | works on tokens, so "token X followed by token Y" rules are easy and readable | works on raw characters; spacing around symbols has to be handled (`\s?`) |
| False positives | `like_num` alone also fires on dates, order ids and phone-number fragments, so it needs a context rule (the unit-word check) | similar risk, but a specific pattern shape (`\d+ unit`) limits it |
| Portability | depends on the tokenizer's decisions | independent of any tokenizer |

In practice the two are complementary: regex is precise for fixed formats (ids, phone numbers), token attributes are better for language-like values (number words, currency symbols) and for rules that use neighbouring tokens.

## Activity 3: Tokenization with NLTK vs spaCy

In [11]:
DOCUMENT = '''"Don't worry," said Dr. O'Brien, "we can't fix it today, but we'll ship the e-mail template by 5 p.m. It's a state-of-the-art tool that costs $5.99/month, roughly 10% cheaper than last year's model! Visit https://www.example.com/help or write to help@example.com for U.S. support (open 9-5). I'm sure y'all won't mind; they've waited 3.5 hours already... Let's go!"'''

nltk_tokens = word_tokenize(DOCUMENT)
spacy_tokens = [t.text for t in nlp(DOCUMENT)]

print("NLTK  :", len(nltk_tokens), "tokens")
print(nltk_tokens)
print()
print("spaCy :", len(spacy_tokens), "tokens")
print(spacy_tokens)

NLTK  : 87 tokens
['``', 'Do', "n't", 'worry', ',', "''", 'said', 'Dr.', "O'Brien", ',', '``', 'we', 'ca', "n't", 'fix', 'it', 'today', ',', 'but', 'we', "'ll", 'ship', 'the', 'e-mail', 'template', 'by', '5', 'p.m', '.', 'It', "'s", 'a', 'state-of-the-art', 'tool', 'that', 'costs', '$', '5.99/month', ',', 'roughly', '10', '%', 'cheaper', 'than', 'last', 'year', "'s", 'model', '!', 'Visit', 'https', ':', '//www.example.com/help', 'or', 'write', 'to', 'help', '@', 'example.com', 'for', 'U.S.', 'support', '(', 'open', '9-5', ')', '.', 'I', "'m", 'sure', "y'all", 'wo', "n't", 'mind', ';', 'they', "'ve", 'waited', '3.5', 'hours', 'already', '...', 'Let', "'s", 'go', '!', "''"]

spaCy : 95 tokens
['"', 'Do', "n't", 'worry', ',', '"', 'said', 'Dr.', "O'Brien", ',', '"', 'we', 'ca', "n't", 'fix', 'it', 'today', ',', 'but', 'we', "'ll", 'ship', 'the', 'e', '-', 'mail', 'template', 'by', '5', 'p.m.', 'It', "'s", 'a', 'state', '-', 'of', '-', 'the', '-', 'art', 'tool', 'that', 'costs', '$', '5.99

In [12]:
# Align the two token streams and print every region where they disagree
sm = difflib.SequenceMatcher(None, nltk_tokens, spacy_tokens, autojunk=False)
rows = []
for tag, i1, i2, j1, j2 in sm.get_opcodes():
    if tag != "equal":
        rows.append((" ".join(nltk_tokens[i1:i2]) or "-", " ".join(spacy_tokens[j1:j2]) or "-"))

print(f"Identical tokens: {sum(b.size for b in sm.get_matching_blocks())}   "
      f"Regions that differ: {len(rows)}\n")
print_table(rows, ["NLTK", "spaCy"])

Identical tokens: 70   Regions that differ: 12

NLTK                            spaCy                       
------------------------------  ----------------------------
``                              "                           
''                              "                           
``                              "                           
e-mail                          e - mail                    
p.m .                           p.m.                        
state-of-the-art                state - of - the - art      
5.99/month                      5.99 / month                
https : //www.example.com/help  https://www.example.com/help
help @ example.com              help@example.com            
9-5                             9 - 5                       
y'all                           y' all                      
''                              "                           


In [13]:
# Targeted probes: how does each tokenizer treat specific constructs?
probes = [
    # contractions
    "don't", "can't", "won't", "I'm", "they've", "let's", "y'all", "O'Brien", "last year's",
    # punctuation / symbols
    '"quoted"', "wait...", "end.", "(brackets)", "state-of-the-art", "e-mail", "9-5",
    # abbreviations and numbers
    "the U.S. economy", "Dr. Smith left", "at 5 p.m. today", "$5.99/month", "10%", "3.5",
    # web
    "https://www.example.com/help", "help@example.com",
]

rows = []
for p in probes:
    n = word_tokenize(p)
    s = [t.text for t in nlp(p)]
    rows.append((p, n, s, "same" if n == s else "DIFFERENT"))

print_table(rows, ["input", "NLTK", "spaCy", "verdict"])

input                         NLTK                                      spaCy                                         verdict  
----------------------------  ----------------------------------------  --------------------------------------------  ---------
don't                         ['do', "n't"]                             ['do', "n't"]                                 same     
can't                         ['ca', "n't"]                             ['ca', "n't"]                                 same     
won't                         ['wo', "n't"]                             ['wo', "n't"]                                 same     
I'm                           ['I', "'m"]                               ['I', "'m"]                                   same     
they've                       ['they', "'ve"]                           ['they', "'ve"]                               same     
let's                         ['let', "'s"]                             ['let', "'s"]                   

In [14]:
# Summary of differences by category (counts over the whole document)
def count(tokens, pred):
    return sum(1 for t in tokens if pred(t))

categories = [
    ("tokens that are punctuation only",       lambda t: all(not c.isalnum() for c in t)),
    ("tokens starting with an apostrophe",     lambda t: t.startswith("'") and len(t) > 1),
    ("tokens equal to n't",                    lambda t: t == "n't"),
    ("quote tokens (`` and '')",               lambda t: t in ("``", "''")),
    ("tokens containing a hyphen",             lambda t: "-" in t and len(t) > 1),
    ("tokens containing @ or ://",             lambda t: "@" in t or "://" in t),
]
rows = [(name, count(nltk_tokens, f), count(spacy_tokens, f)) for name, f in categories]
rows.append(("TOTAL tokens", len(nltk_tokens), len(spacy_tokens)))
print_table(rows, ["category", "NLTK", "spaCy"])

category                            NLTK  spaCy
----------------------------------  ----  -----
tokens that are punctuation only    20    23   
tokens starting with an apostrophe  8     6    
tokens equal to n't                 3     3    
quote tokens (`` and '')            4     0    
tokens containing a hyphen          3     0    
tokens containing @ or ://          1     2    
TOTAL tokens                        87    95   


### Observations (Activity 3)

**Identical:** contractions are split the same way by both tokenizers (`Do|n't`, `ca|n't`, `wo|n't`, `'ll`, `'s`, `'m`, `'ve`) and `O'Brien`, `3.5`, `10 %`, `...` and `U.S.` / `Dr.` inside a sentence are handled the same.

**Different:**

| Construct | NLTK `word_tokenize` | spaCy |
|---|---|---|
| straight double quotes `"` | converted to `` ` ` `` and `''` (Penn Treebank style) | kept as `"` |
| hyphenated words (`e-mail`, `state-of-the-art`, `9-5`) | kept as one token | split at every hyphen |
| `5.99/month` | one token `5.99/month` | `5.99`, `/`, `month` |
| URLs and emails | split at `:` and `@` (`https`, `:`, `//www...`, `help`, `@`, `example.com`) | kept as a single token |
| `y'all` | one token | `y'`, `all` |
| abbreviation at the end of a sentence (`p.m.` followed by a new sentence) | the final period is split off (`p.m`, `.`) | `p.m.` kept whole |

NLTK first splits the text into sentences and only then tokenizes, so the treatment of a trailing period depends on whether Punkt decided the abbreviation ended a sentence; spaCy decides by tokenizer exception rules that do not depend on the surrounding sentence. The probe table above shows the contextual cases (`the U.S. economy`, `Dr. Smith left`) for that reason. Overall spaCy produced more tokens than NLTK on this document, mainly because it splits at hyphens and slashes.

## Activity 4: Sentence segmentation with NLTK and spaCy, and mismatch flagging

The paragraph is deliberately full of full stops that do **not** end a sentence (titles, abbreviations, decimals, a quoted sentence, `a.m.`). The list `GOLD` holds the six sentences as a human would split them, so each tool can also be scored against it.

In [15]:
GOLD = [
    "Dr. Smith moved to the U.S. in 2015 and joined Acme Inc. as a data scientist.",
    "She earned her Ph.D. at MIT, which she often calls 'the best 4.5 years of my life'.",
    "Did you know that e.g. tokenization is hard?",
    'Mr. Jones said, "It costs $3.50 per unit."',
    "The meeting starts at 10 a.m. on Monday.",
    "Please arrive early!",
]
PARAGRAPH = " ".join(GOLD)
print(PARAGRAPH)
print("\nExpected number of sentences:", len(GOLD))

Dr. Smith moved to the U.S. in 2015 and joined Acme Inc. as a data scientist. She earned her Ph.D. at MIT, which she often calls 'the best 4.5 years of my life'. Did you know that e.g. tokenization is hard? Mr. Jones said, "It costs $3.50 per unit." The meeting starts at 10 a.m. on Monday. Please arrive early!

Expected number of sentences: 6


In [16]:
# --- NLTK: Punkt sentence tokenizer (unsupervised, knows many abbreviations)
nltk_sents = sent_tokenize(PARAGRAPH)

# --- spaCy: rule-based sentencizer (splits after sentence-final punctuation tokens)
nlp_sent = spacy.blank("en")
nlp_sent.add_pipe("sentencizer")
spacy_doc = nlp_sent(PARAGRAPH)
spacy_sents = [s.text for s in spacy_doc.sents]

print(f"NLTK  -> {len(nltk_sents)} sentences")
for i, s in enumerate(nltk_sents, 1):
    print(f"  {i}. {s}")
print(f"\nspaCy -> {len(spacy_sents)} sentences")
for i, s in enumerate(spacy_sents, 1):
    print(f"  {i}. {s}")

NLTK  -> 7 sentences
  1. Dr. Smith moved to the U.S. in 2015 and joined Acme Inc. as a data scientist.
  2. She earned her Ph.D. at MIT, which she often calls 'the best 4.5 years of my life'.
  3. Did you know that e.g.
  4. tokenization is hard?
  5. Mr. Jones said, "It costs $3.50 per unit."
  6. The meeting starts at 10 a.m. on Monday.
  7. Please arrive early!

spaCy -> 6 sentences
  1. Dr. Smith moved to the U.S. in 2015 and joined Acme Inc. as a data scientist.
  2. She earned her Ph.D. at MIT, which she often calls 'the best 4.5 years of my life'.
  3. Did you know that e.g. tokenization is hard?
  4. Mr. Jones said, "It costs $3.50 per unit."
  5. The meeting starts at 10 a.m. on Monday.
  6. Please arrive early!


In [17]:
def start_offsets(sentences, text):
    # character offset where each sentence begins in the paragraph
    offsets, pos = [], 0
    for s in sentences:
        idx = text.find(s, pos)
        offsets.append(idx)
        pos = idx + len(s)
    return offsets


nltk_starts  = set(start_offsets(nltk_sents, PARAGRAPH))
spacy_starts = set(start_offsets(spacy_sents, PARAGRAPH))
gold_starts  = set(start_offsets(GOLD, PARAGRAPH))


def snippet(offset, width=22):
    left = PARAGRAPH[max(0, offset - width):offset]
    right = PARAGRAPH[offset:offset + width]
    return f"...{left}[|]{right}..."


print("Sentence-boundary comparison (a boundary = start of a sentence)\n")
only_nltk  = sorted(nltk_starts - spacy_starts)
only_spacy = sorted(spacy_starts - nltk_starts)

if not only_nltk and not only_spacy:
    print("NO MISMATCH: both tools produced exactly the same boundaries.")
else:
    print(f"MISMATCH FLAGGED: {len(only_nltk) + len(only_spacy)} boundary difference(s)\n")
    for off in only_spacy:
        print(f"  [spaCy only] split at offset {off:>3}: {snippet(off)}")
    for off in only_nltk:
        print(f"  [NLTK only]  split at offset {off:>3}: {snippet(off)}")

Sentence-boundary comparison (a boundary = start of a sentence)

MISMATCH FLAGGED: 1 boundary difference(s)

  [NLTK only]  split at offset 185: ...id you know that e.g. [|]tokenization is hard? ...


In [18]:
# Sentence-by-sentence view, flagging rows that disagree
# (aligned with difflib so that one extra split does not make every later sentence look different)
sm_s = difflib.SequenceMatcher(None, nltk_sents, spacy_sents, autojunk=False)
for tag, i1, i2, j1, j2 in sm_s.get_opcodes():
    if tag == "equal":
        for s in nltk_sents[i1:i2]:
            print(f"[ok]        {s}")
    else:
        print("[MISMATCH]")
        for s in nltk_sents[i1:i2]:
            print(f"   NLTK : {s}")
        for s in spacy_sents[j1:j2]:
            print(f"   spaCy: {s}")

# Scores against the human split
def boundary_scores(pred, gold):
    tp = len(pred & gold)
    p = tp / len(pred) if pred else 0.0
    r = tp / len(gold) if gold else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return round(p, 4), round(r, 4), round(f, 4)

print()
print_table([
    ("NLTK Punkt",         len(nltk_sents),  *boundary_scores(nltk_starts,  gold_starts)),
    ("spaCy sentencizer",  len(spacy_sents), *boundary_scores(spacy_starts, gold_starts)),
], ["tool", "#sentences", "precision", "recall", "F1"])

[ok]        Dr. Smith moved to the U.S. in 2015 and joined Acme Inc. as a data scientist.
[ok]        She earned her Ph.D. at MIT, which she often calls 'the best 4.5 years of my life'.
[MISMATCH]
   NLTK : Did you know that e.g.
   NLTK : tokenization is hard?
   spaCy: Did you know that e.g. tokenization is hard?
[ok]        Mr. Jones said, "It costs $3.50 per unit."
[ok]        The meeting starts at 10 a.m. on Monday.
[ok]        Please arrive early!

tool               #sentences  precision  recall  F1    
-----------------  ----------  ---------  ------  ------
NLTK Punkt         7           0.8571     1.0     0.9231
spaCy sentencizer  6           1.0        1.0     1.0   


In [19]:
# Optional: spaCy's statistical (dependency-parser based) sentence splitter, if a model is installed
try:
    nlp_parser = spacy.load("en_core_web_sm")
    parser_sents = [s.text for s in nlp_parser(PARAGRAPH).sents]
    print(f"spaCy parser -> {len(parser_sents)} sentences")
    for i, s in enumerate(parser_sents, 1):
        print(f"  {i}. {s}")
    parser_starts = set(start_offsets(parser_sents, PARAGRAPH))
    print("\nBoundaries identical to NLTK:", parser_starts == nltk_starts)
except OSError:
    print("en_core_web_sm is not installed; skipping this optional comparison.")
    print("Install it with:  python -m spacy download en_core_web_sm")

en_core_web_sm is not installed; skipping this optional comparison.
Install it with:  python -m spacy download en_core_web_sm


## Summary

- **Regex** is the right tool for fixed formats (emails, ids, phone numbers) but each format must be spelled out in the pattern.
- **Token attributes** (`like_num`, `is_currency`, ...) handle number formats and symbols for free, but need a context rule (here, a following unit word) to avoid flagging dates, ids and phone fragments as quantities.
- **NLTK vs spaCy tokenization** agree on most tokens, and differ mainly in punctuation (quotes, ellipses, hyphens, abbreviations at the end of a sentence), URLs/emails and a few contractions.
- **Sentence segmentation:** on the test paragraph the rule-based spaCy sentencizer reproduced the human split, because spaCy's tokenizer keeps `Dr.`, `U.S.`, `Inc.`, `Ph.D.`, `e.g.` and `a.m.` as single tokens and the sentencizer only breaks after standalone sentence-final punctuation. NLTK's Punkt handled the titles and abbreviations at most positions but split after `e.g.` when it was followed by a lower-case word, giving 7 sentences instead of 6; the boundary comparison flags exactly that disagreement. Which tool is "right" depends on the text: Punkt is trained on abbreviation statistics, the sentencizer is purely rule-based, and a parser-based spaCy model (optional cell) can use syntax.